In [ ]:
#This houses the code used to create the normalized_health_dataset
import pandas as pd
from datascience import *

#Transforms strings into numeric
from sklearn.preprocessing import LabelEncoder

#Define dataset
health_db = pd.read_csv("health_dataset.csv")

In [ ]:
#Transform string values into numeric using label encoding
str_columns = ["student_id", "gender", "part_time_job", "diet_quality", "parental_education_level",
           "internet_quality", "extracurricular_participation"]
for feature in str_columns:
    le = LabelEncoder()
    health_db[feature] = le.fit_transform(health_db[feature])

health_db.info() #Show column values
health_db.head(5) #Show dataset structure

In [ ]:
#Dataset is not consistent enough for the model. This block will create a function
#that will treat each column as a question that impact the final rating.

#Normalization function
def value_normalize(value, min_value, max_value):
    """Function to normalize numeric values.
    Inputs:
        value: Column referneced
        min_value: Minimum value of column
        max_value: Maximum value of column
    Returns:
        normalize (decimal): Normalized value 
    """
    #Equation for normalization
    normalize = (value - min_value) / (max_value - min_value)

    #Return new value
    #Also caps the value from 0-1. No outliers
    return max(0, min (1, normalize))


#Consistant score calculator function
#More columns can be added in the future if needed
def score_calculator(row):
    """Function to calculate the mental_health_score of every row through normalization. This is to 
    replace inconsistant scoring with calculated scoring based on the same weights.
    Inputs:
        row: Each row in health_db
    Returns:
        final_score: Score based off all normalized columns that will be used
    """
    #Normalizing each column that will be used. row, min, max structure
    sleep_norm = value_normalize(row["sleep_hours"], 4, 8)
    study_norm = value_normalize(row["study_hours_per_day"], 0, 8)
    social_media_norm = value_normalize(row["social_media_hours"], 0, 8)
    tv_norm = value_normalize(row["netflix_hours"], 0, 6)
    job_norm = value_normalize(row["part_time_job"], 0, 1)
    diet_quality_norm = value_normalize(row["diet_quality"], 0, 2)
    exercise_norm = value_normalize(row["exercise_frequency"], 0, 5)
    extra_norm = value_normalize(row["extracurricular_participation"], 0, 1)
    
    #Weight Application to determine final score (Combined they all add to 1.0)
    #Final score equation (w1x1 + ... w?x?)
    final_score = (0.15*exercise_norm + 0.2*sleep_norm + 0.1*study_norm + 
                   0.15*social_media_norm + 0.1*job_norm + 0.1*diet_quality_norm +
                   0.1*tv_norm + 0.1*extra_norm)

    #Return final result
    return final_score

#Drop unused columns (Can be changed later)
health_db = health_db.drop(["student_id", "gender", "age", "parental_education_level", 
                            "attendance_percentage", "internet_quality", "exam_score"], axis = 1)

#Replace current ratings with new the ones calculated
health_db['mental_health_rating'] = health_db.apply(score_calculator, axis = 1)

#Save data to new csv
health_db.to_csv("normalized_health_dataset.csv", index = False)
    